In [4]:
import pandas as pd

df = pd.read_parquet('df_fwd_full.parquet')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 10 rows:")
print(df.head(10))
print("\nDate range:")
print(f"From {df['trade_date'].min() if 'trade_date' in df.columns else df['date'].min()}")
print(f"To {df['trade_date'].max() if 'trade_date' in df.columns else df['date'].max()}")
print("\nUnique DTEs:", df['dte'].unique() if 'dte' in df.columns else 'N/A')

Shape: (2293082, 27)

Columns: ['trade_date', 'strike', 'expiration', 'option_type', 'last', 'bid', 'ask', 'bid_iv', 'ask_iv', 'open_interest', 'volume', 'delta', 'gamma', 'vega', 'theta', 'rho', 'dte', 'ticker', 'year', 'spx_close', 'T', 'df', 'r', 'forward', 'n_strikes', 'iv_mid', 'log_moneyness_fwd']

First 10 rows:
  trade_date  strike expiration option_type    last     bid     ask  bid_iv  \
0 2010-01-04  1000.0 2010-01-08           C  132.21  130.90  133.90  0.0000   
1 2010-01-04  1025.0 2010-01-08           C  107.21  106.00  109.00  0.0000   
2 2010-01-04  1050.0 2010-01-08           C   82.23   81.20   84.20  0.0000   
3 2010-01-04  1075.0 2010-01-08           C   57.44   57.00   58.80  0.0000   
4 2010-01-04  1100.0 2010-01-08           C   33.58   33.00   34.40  0.1942   
5 2010-01-04  1125.0 2010-01-08           C   12.51   11.40   12.70  0.1534   
6 2010-01-04  1150.0 2010-01-08           C    1.28    1.60    1.70  0.1528   
7 2010-01-04  1175.0 2010-01-08           C    

In [12]:
import pandas as pd
import numpy as np

df = pd.read_parquet('df_fwd_full.parquet')
df = df[df['option_type'] == 'C'].copy()

# ── 30 DTE leg ──────────────────────────────────────────────
df_30 = df[(df['dte'] >= 20) & (df['dte'] <= 40)].copy()
df_30['dte_diff'] = abs(df_30['dte'] - 30)
df_30['moneyness'] = abs(df_30['strike'] - df_30['spx_close'])

# Sort so best option is first: closest DTE, then closest strike
df_30 = df_30.sort_values(['trade_date', 'dte_diff', 'moneyness'])
df_30_atm = df_30.drop_duplicates(subset='trade_date', keep='first')

# ── 60 DTE leg ──────────────────────────────────────────────
df_60 = df[(df['dte'] >= 50) & (df['dte'] <= 70)].copy()
df_60['dte_diff'] = abs(df_60['dte'] - 60)
df_60['moneyness'] = abs(df_60['strike'] - df_60['spx_close'])

df_60 = df_60.sort_values(['trade_date', 'dte_diff', 'moneyness'])
df_60_atm = df_60.drop_duplicates(subset='trade_date', keep='first')

print(f"30 DTE days: {len(df_30_atm)}")
print(f"60 DTE days: {len(df_60_atm)}")

# ── Merge ────────────────────────────────────────────────────
df_calendar = pd.merge(
    df_30_atm[['trade_date', 'strike', 'bid', 'ask', 'iv_mid', 'dte', 'spx_close']],
    df_60_atm[['trade_date', 'strike', 'bid', 'ask', 'iv_mid', 'dte']],
    on='trade_date',
    suffixes=('_30', '_60'),
    how='inner'
)

df_calendar['option_30dte_mid'] = (df_calendar['bid_30'] + df_calendar['ask_30']) / 2
df_calendar['option_60dte_mid'] = (df_calendar['bid_60'] + df_calendar['ask_60']) / 2
df_calendar['option_30dte_bid'] = df_calendar['bid_30']
df_calendar['option_30dte_ask'] = df_calendar['ask_30']
df_calendar['option_60dte_bid'] = df_calendar['bid_60']
df_calendar['option_60dte_ask'] = df_calendar['ask_60']

df_calendar = df_calendar.rename(columns={
    'trade_date': 'date',
    'spx_close': 'spx_price',
    'strike_30': 'strike'
})

final_columns = [
    'date', 'spx_price', 'strike',
    'option_30dte_bid', 'option_30dte_mid', 'option_30dte_ask',
    'option_60dte_bid', 'option_60dte_mid', 'option_60dte_ask'
]

df_simulation_ready = df_calendar[final_columns].copy()

print(f"\nShape: {df_simulation_ready.shape}")
print(f"Date range: {df_simulation_ready['date'].min()} to {df_simulation_ready['date'].max()}")

gaps = df_simulation_ready['date'].diff().dt.days
print(f"Gaps > 1 week: {(gaps > 7).sum()}")
print(f"Gaps > 3 days: {(gaps > 3).sum()}")

df_simulation_ready.to_parquet('calendar_spread_data.parquet')
print("\nSaved!")

30 DTE days: 3899
60 DTE days: 3801

Shape: (3727, 9)
Date range: 2010-01-11 00:00:00 to 2025-12-31 00:00:00
Gaps > 1 week: 42
Gaps > 3 days: 150

Saved!
